# Judge Reliability — oracle ICC + second-judge agreement  `[MEASUREMENT VALIDITY]`

Buys down **LIMITATIONS.md §1** (judge reliability is not measured) and **§2** (patient = oracle coupling) on a
cheap SUBSET. Two parts, each behind an explicit `RUN_*` flag so opening this notebook never spends money:

1. **Repeatability (ICC)** — re-score the anchor models' conversations `N_REPS` times with the PRIMARY oracle
   (gpt-4o-mini, same temperature, per-rep seeds — the main pipeline pins `seed=42`, so its reps would be
   trivially identical). → per-metric **ICC(2,1)** + mean |Δ|, upgrading the "oracle noise ≈ 0.10" folklore into a citable statistic.
2. **Second judge** — score the same conversations once with a **different judge** (pluggable: OpenAI or **Claude**
   via the `anthropic` SDK). → per-metric agreement (r/ρ/bias) vs the primary oracle **and the defense-critical check**:
   does the PTO-vs-GRPO endpoint contrast survive a judge that did NOT simulate the patient?

Outputs land in `data/judge_check/` (never the real `eval_scores/`); scoring is resume-safe (existing CSVs skipped).
Summary tables are written to `data/judge_check/summary/` — surface them in `LIMITATIONS.md` / `4_Training_and_Reliability`
once real numbers exist. This notebook is a scoring pipeline like `Run_Eval.ipynb` — it is NOT part of `render_views.py`.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import asyncio
import numpy as np, pandas as pd
pd.set_option("display.width", 185, "display.max_columns", 50)
from eda_analysis.scoring import registry as osc_config, conversations as osc_data, judge as jc

# ╔═══ KNOBS ═══════════════════════════════════════════════════════════════════╗
# Anchor model states (base + endpoints + the GRPO peak). file_index pairing across arms is
# persona-valid ONLY at matched iterations (same seed+k+1 shuffle), which these are.
SUBSET_MODELS = ["PTOExp3_LA0_Base", "PTOExp3_LA0_I10", "GRPOExp3_LA0_I8", "GRPOExp3_LA0_I10"]
METRICS   = ["Q1", "Q2", "MICI"]   # Q1+Q2 = the headline reward metric; MICI = the sycophancy claim
SUBSET_N  = None                     # None = all 96 convs per model; or an int to shrink further
N_REPS    = 3                        # ICC repetitions with the primary oracle
CONCURRENCY = 16

# The second judge — LIOR DECIDES THE MODEL. Claude options (input/output $ per MTok):
#   claude-haiku-4-5  ($1/$5, economical default) · claude-sonnet-5 ($3/$15) · claude-opus-4-8 ($5/$25)
# Requires `pip install anthropic` + a key in env ANTHROPIC_API_KEY or anthropic_key.txt at the
# experiment root (beside openai_key.txt). Provider "openai" also works (e.g. a different GPT).
#
# ⚠ SWAPPING TO SONNET 5 / OPUS 4.8+ ALSO NEEDS `thinking={"type": "disabled"}`: those models run
#   ADAPTIVE thinking when `thinking` is omitted, and the thinking tokens bill against the same
#   `max_tokens` (1024) → truncated JSON + unpredictable cost. Haiku 4.5 needs no thinking config.
#     SECOND_JUDGE = jc.JudgeSpec(provider="anthropic", model="claude-sonnet-5",
#                                 thinking={"type": "disabled"})
SECOND_JUDGE = jc.JudgeSpec(provider="anthropic", model="claude-haiku-4-5")

# Money switches — nothing is scored until you flip these to True and run the cell below.
RUN_ICC          = False
RUN_SECOND_JUDGE = False
# ── FULL dual-judge sweep (section 3) ─────────────────────────────────────────
# Promotes the second judge from a spot check to a co-primary measurement: ALL model states x ALL
# 8 rubrics. Batched (50% off) because this feeds an offline EDA and has no latency requirement.
FULL_METRICS = ["Q1", "Q2", "WAI-SR", "CSQ-8", "MI-SAT", "MITI", "PCT", "MICI"]
FULL_REP     = 0
PROBE_USAGE  = False   # True = spend ~$0.02 on real calls to MEASURE tokens instead of estimating
RUN_FULL_SWEEP     = False   # master switch for section 3c
FULL_SWEEP_DRY_RUN = True    # True = build + count requests, submit NOTHING (set False to spend)
# ╚═════════════════════════════════════════════════════════════════════════════╝

PRIMARY = jc.PRIMARY_JUDGE
print("primary judge:", PRIMARY.tag, "| second judge:", SECOND_JUDGE.tag)
print("judge_check root:", jc.JUDGE_CHECK_ROOT)

In [ ]:
# Load the subset conversations (from the auto-discovered registry) + preview the call count/cost.
exps = [e for e in osc_config.EXPERIMENTS if e.model_name in SUBSET_MODELS]
assert {e.model_name for e in exps} == set(SUBSET_MODELS), \
    f"missing on disk: {set(SUBSET_MODELS) - {e.model_name for e in exps}} (Drive symlinks mounted?)"
paths  = osc_config.resolve_paths(exps)
names  = [e.model_name for e in exps]
layout = osc_config.get_model_eval_layout(exps)
combined = osc_data.combine_data(osc_data.load_data(paths), names)
n_convs = combined.groupby("Model").size().min() if SUBSET_N is None else SUBSET_N
print(combined.groupby("Model").size().rename("convs on disk"))

# Cost preview (rough: ~2.5k input + ~0.15k output tokens per oracle call).
calls_icc   = len(SUBSET_MODELS) * n_convs * len(METRICS) * N_REPS
calls_judge = len(SUBSET_MODELS) * n_convs * len(METRICS)
IN_TOK, OUT_TOK = 2500, 150
price = {"gpt-4o-mini": (0.15, 0.60), "claude-haiku-4-5": (1.0, 5.0),
         "claude-sonnet-5": (3.0, 15.0), "claude-opus-4-8": (5.0, 25.0)}
def est(n_calls, model_key):
    pin, pout = next(v for k, v in price.items() if k in model_key)
    return n_calls * (IN_TOK * pin + OUT_TOK * pout) / 1e6
print(f"\nICC: {calls_icc} calls with {PRIMARY.model} ≈ ${est(calls_icc, 'gpt-4o-mini'):.2f}")
try:
    print(f"2nd judge: {calls_judge} calls with {SECOND_JUDGE.model} ≈ ${est(calls_judge, SECOND_JUDGE.model):.2f}")
except StopIteration:
    print(f"2nd judge: {calls_judge} calls with {SECOND_JUDGE.model} (add its price to `price` for an estimate)")

## 1 · Repeatability — primary oracle × N_REPS  `[$ — flips on RUN_ICC]`
Rep 0 deliberately mirrors the production configuration; reps differ only in the API `seed` (temperature stays 0.1),
so the ICC captures pure sampling noise of the measurement instrument.

In [ ]:
if RUN_ICC:
    for rep in range(N_REPS):
        await jc.run_judge_scoring(PRIMARY, combined, METRICS, layout,
                                   rep=rep, concurrency=CONCURRENCY, subset_n=SUBSET_N)
else:
    print("RUN_ICC=False — skipping (flip the knob in cell 1 to score).")

In [ ]:
# ICC(2,1) + mean |Δ| between reps, per (metric, model) — runs on whatever is on disk.
icc_long = jc.load_judge_scores(PRIMARY.tag)
if icc_long.empty or icc_long.rep.nunique() < 2:
    print("Need ≥2 scored reps for ICC — nothing to report yet.")
    REP_TAB = None
else:
    REP_TAB = jc.repeatability_table(icc_long)
    display(REP_TAB)
    print("Read: ICC(2,1) ≥ 0.75 = good, ≥ 0.9 = excellent test-retest reliability (Koo & Li 2016 guidelines);",
          "mean_abs_diff is the per-conversation |Δ| between reps (compare to the ~0.10 folklore figure).")

## 2 · Second judge — decoupled from the patient simulator  `[$ — flips on RUN_SECOND_JUDGE]`
The simulated patient and the primary grader are the same model (gpt-4o-mini). A second judge from a **different
family** breaks that coupling. Two questions, in order of importance: **(a)** does the PTO-vs-GRPO endpoint contrast
survive? **(b)** how well do the per-conversation scores agree (r / ρ / bias)?

In [ ]:
if RUN_SECOND_JUDGE:
    await jc.run_judge_scoring(SECOND_JUDGE, combined, METRICS, layout,
                               rep=0, concurrency=CONCURRENCY, subset_n=SUBSET_N)
else:
    print("RUN_SECOND_JUDGE=False — skipping (flip the knob in cell 1 to score).")

In [ ]:
# Primary-oracle scores for the same (model, metric, conversation) cells, from the REAL eval_scores tree.
def load_primary_long(models, metrics):
    rows = []
    for model in models:
        entry = layout[model]
        for name in metrics:
            sub, col = jc.JUDGE_METRIC_COLS[name]
            ddir = osc_config.eval_csv_dir(entry["root"], entry["oracle"], sub, model)
            if not os.path.isdir(ddir):
                continue
            for fn in os.listdir(ddir):
                stem, ext = os.path.splitext(fn)
                if ext != ".csv" or not stem.isdigit():
                    continue
                try:
                    df = pd.read_csv(os.path.join(ddir, fn))
                except Exception:
                    continue
                if len(df) and col in df.columns:
                    rows.append({"metric": name, "model": model, "file_index": int(stem),
                                 "value": float(df[col].iloc[0])})
    return pd.DataFrame(rows)

judge_long   = jc.load_judge_scores(SECOND_JUDGE.tag, reps=[0])
primary_long = load_primary_long(SUBSET_MODELS, METRICS)
if judge_long.empty:
    print("Second judge has no scores on disk yet.")
    AGR_TAB = None; CONTRASTS = None
else:
    AGR_TAB = jc.agreement_table(judge_long, primary_long)
    display(AGR_TAB)
    # THE defense check: is the endpoint contrast judge-independent?
    CONTRASTS = pd.DataFrame([
        jc.contrast_preservation(judge_long, primary_long, "PTOExp3_LA0_I10", "GRPOExp3_LA0_I10", m)
        for m in METRICS
    ] + [
        jc.contrast_preservation(judge_long, primary_long, "PTOExp3_LA0_I10", "PTOExp3_LA0_Base", m)
        for m in METRICS
    ])
    display(CONTRASTS)
    print("Read: `same_sign=True` on the PTO−GRPO endpoint rows = the headline result is not an artifact",
          "of the shared patient/oracle model. For MICI remember lower = better (a positive Δ is worse).")

## 3 · FULL dual-judge sweep — every model × every rubric  `[$$ — flips on RUN_FULL_SWEEP]`

§1–§2 measured the instrument on a 4-model × 3-metric anchor subset. This section promotes the second judge from a *spot check* to a **co-primary measurement**: all 29 model states × all 8 rubrics × 96 conversations, so every number the thesis reports has a held-out-judge counterpart.

**The framing that matters.** The two judges are *not* interchangeable raters to be averaged. The primary oracle (`gpt-4o-mini` on Q1+Q2) **was the training reward** — both methods optimized it directly. The second judge never touched training. So this is an **optimization-target vs held-out-test** comparison, and averaging them would be like averaging train and test accuracy. Nothing downstream ever averages raw scores across judges; only contrasts and standardized quantities are compared. (Level bias runs 1.2–1.7 points and is *model-dependent*, comparable in size to the headline effect itself.)

**Four phases, three of them free:**

| Phase | Cost | What it does |
|---|---|---|
| **3a Parity gate** | free | Claude's `json_schema` rejects `minimum`/`maxItems`/…, so those are folded into `description`. This verifies every dropped constraint was restated and the two encodings are structurally identical. **A red row here means the judges would be answering different rubrics — do not proceed.** |
| **3b Plan + cost** | free | Coverage-aware call count (skips what is already on disk) × measured token usage → projected USD, batched and not. |
| **3c Submit** | **$$** | Anthropic Message Batches (**50% off**; no latency requirement here, so this is free money). `dry_run=True` by default. |
| **3d Collect** | free | Poll, then write per-conversation CSVs in the standard `judge_check/` layout. Resume-safe: submit and collect are separate calls, so a closed laptop or a fresh kernel loses nothing. |

**On prompt caching — measured, not assumed.** `prefix_report()` shows only **Q1 (~1.1k tok) and Q2 (~2.2k tok)** clear OpenAI's 1,024-token cacheable-prefix minimum. WAI-SR/CSQ-8/MI-SAT are correctly rubric-first but simply too short (403–507 tok); MITI/PCT/MICI interpolate a *per-conversation* utterance count into the instructions **before** the rubric, truncating their prefix to 138–206 tok. And Haiku 4.5's cache minimum is **4,096**, so the second judge never caches at all. This is documented rather than fixed: those counts are the denominators the rate metrics need, the prompt is the measurement instrument, and changing it would break comparability with all 22,272 conversations already scored — for a discount that still would not materialize.

**On repetitions.** Extra reps are deliberately *not* bought here. Oracle noise contributes ≈0.01 to a 96-conversation arm mean against ≈0.09 from persona sampling — an order of magnitude less — so a second rep cannot move any arm-level conclusion. Breadth (all models × both judges) dominates depth (more reps of a few cells) at equal cost. The one exception is **MICI**, whose ICC (0.89–0.96) is the lowest and where the judges genuinely diverge; 2–3 reps there are defensible and cheap on the existing live path in §1.

In [ ]:
from eda_analysis.scoring import judge_plan as jp, judge_batch as jb

# ── 3a · PARITY GATE (free) — must be all-True before any spend ───────────────
PARITY = jp.check_rubric_parity(FULL_METRICS)
display(PARITY)
assert PARITY.parity_ok.all(), "RUBRIC PARITY FAILED — the two judges would answer different " \
                               "rubrics. Fix scoring/judge.py::_strip_unsupported_constraints " \
                               "before spending anything."
print("parity OK — both judges receive the same rubric for all", len(PARITY), "questionnaires.\n")

# Which rubrics actually get a prompt-cache discount, measured rather than assumed.
display(jp.prefix_report(FULL_METRICS)[
    ["metric", "prefix_tokens_approx", "caches_on_gpt-4o-mini-2024-07-18", "caches_on_claude-haiku-4-5"]])

# ── 3b · PLAN + COST (free) ───────────────────────────────────────────────────
FULL_EXPS   = list(osc_config.EXPERIMENTS)
FULL_NAMES  = [e.model_name for e in FULL_EXPS]
FULL_LAYOUT = osc_config.get_model_eval_layout(FULL_EXPS)
FULL_COMBINED = osc_data.combine_data(osc_data.load_data(osc_config.resolve_paths(FULL_EXPS)),
                                      FULL_NAMES)
print(f"{len(FULL_NAMES)} model states, {len(FULL_COMBINED):,} conversations on disk")

PLAN = jp.plan_sweep(SECOND_JUDGE, FULL_COMBINED, FULL_METRICS, FULL_LAYOUT, rep=FULL_REP)
print(PLAN)

# Token usage: MEASURED if you can spare ~$0.02, else estimated from transcript lengths.
if PROBE_USAGE:
    USAGE = await jb.probe_usage(SECOND_JUDGE, FULL_COMBINED, FULL_METRICS, n_per_metric=2)
else:
    USAGE = jp.usage_from_chars(FULL_COMBINED, FULL_METRICS, model_name=SECOND_JUDGE.model)
print("usage profile:", USAGE, "\n")
display(jp.sweep_report(PLAN, USAGE, model_name=SECOND_JUDGE.model))

# ── 3c · SUBMIT (PAID — dry_run=True until you say otherwise) ─────────────────
if RUN_FULL_SWEEP:
    BATCH_IDS = jb.submit_sweep(SECOND_JUDGE, FULL_COMBINED, FULL_METRICS, FULL_LAYOUT,
                                rep=FULL_REP, dry_run=FULL_SWEEP_DRY_RUN)
else:
    print("RUN_FULL_SWEEP=False — nothing submitted.")

# ── 3d · POLL + COLLECT (free; safe to run in a fresh kernel hours later) ──────
STATUS = jb.poll_batches(SECOND_JUDGE, rep=FULL_REP)
if STATUS is not None and not STATUS.empty:
    display(STATUS)
    if (STATUS.status == "ended").any():
        jb.collect_batches(SECOND_JUDGE, rep=FULL_REP)
        display(jp.plan_sweep(SECOND_JUDGE, FULL_COMBINED, FULL_METRICS, FULL_LAYOUT,
                              rep=FULL_REP).cells.groupby("metric")[["n_existing", "n_todo"]].sum())
else:
    print("No batches submitted yet for this judge/rep.")

In [ ]:
# Convenience CSV snapshot next to the raw scores. The TRACKED thesis artifacts (md + xlsx + the
# three figures) are produced by `5_Training_and_Reliability` §7 via `eda_analysis.save_table` /
# `save_fig` — this notebook owns the paid scoring, notebook 5 owns the presentation.
SUMMARY_DIR = os.path.join(jc.JUDGE_CHECK_ROOT, "summary")
os.makedirs(SUMMARY_DIR, exist_ok=True)
def save(df, name):
    if df is None or (hasattr(df, "empty") and df.empty):
        return
    df.to_csv(os.path.join(SUMMARY_DIR, f"{name}.csv"), index=False)
    print("saved", name)
save(globals().get("REP_TAB"), "oracle_repeatability_icc")
save(globals().get("AGR_TAB"), f"second_judge_agreement_{SECOND_JUDGE.tag}")
save(globals().get("CONTRASTS"), f"second_judge_contrasts_{SECOND_JUDGE.tag}")
print("\nNext: run 5_Training_and_Reliability.ipynb (or `python render_views.py --nb 5`) to render "
      "the tables + figures into results/<view>/.")

## How to read / next steps
- **ICC ≥ 0.75** on Q1/Q2 → the per-conversation scores are reliable; the paired 96-persona design then makes the
  arm-level contrasts far more reliable still (means over 96 convs shrink noise ~√96).
- **Contrast preservation** is the load-bearing result for the thesis: if the second judge (different model family,
  never saw the patient role) reproduces the sign of PTO−GRPO at iter 10 and the vs-base gains, LIMITATIONS §2's
  coupling concern is empirically bounded, not just acknowledged.
- Agreement (r/ρ) will be attenuated by both judges' own noise — interpret against the ICC ceiling
  (max expected r ≈ √(ICC_primary × ICC_judge)), not against 1.0.
- **Presentation lives in `5_Training_and_Reliability` §7**, not here: it reads the same `data/judge_check/` tree
  through `eda_analysis.reliability` (no API calls) and exports the tracked tables + figures under
  `results/<view>/…/5_training/`. This notebook is the paid scorer; keep the `RUN_*` switches off by default.
- After running: update **LIMITATIONS.md §1/§2** with the measured numbers.
- Cost levers: `SUBSET_N`, fewer `METRICS`, fewer `SUBSET_MODELS`. The defaults (4 models × 96 convs × 3 metrics)
  are a few dollars on gpt-4o-mini/haiku — see the preview cell.